# 🔍 Optimización de Hiperparámetros — Estrategia S7

## Federated Proactive Forest

**Estrategia:** Per-Client F1 + PCD Progressive Forest. Selección round-robin balanceando Macro-F1 local y diversidad.

**Hiperparámetros optimizados:**
- Definidos dinámicamente en `search_spaces.yaml`.

> 📋 **Configuración:** Los rangos se leen dinámicamente de `configs/optimization/search_spaces.yaml`.

In [ ]:
# ── Configuración Global ──────────────────────────────────────────────────
import sys
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import yaml
import json

# Configuración de rutas
ROOT = Path.cwd().parent.parent.parent.parent
sys.path.insert(0, str(ROOT))

from src.infrastructure.dataset.dataset_factory import DatasetFactory
from src.application.hyperparam_optimizer import HyperparamOptimizer

# Parámetros de la optimización
STRATEGY = 'S7'
N_TRIALS = 20
METRIC = 'macro_f1'
SEED = 42

RESULTS_DIR = ROOT / 'results' / f'optimization_{STRATEGY.lower()}'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Cargar espacio de búsqueda
with open(ROOT / 'configs' / 'optimization' / 'search_spaces.yaml') as f:
    all_spaces = yaml.safe_load(f)
    if STRATEGY not in all_spaces:
        raise ValueError(f"Estrategia {STRATEGY} no encontrada en search_spaces.yaml")
    SEARCH_SPACE = all_spaces[STRATEGY]

def run_optimization(dataset_name: str):
    print(f"\n{'='*70}")
    print(f"🚀 OPTIMIZANDO {STRATEGY} — DATASET: {dataset_name.upper()}")
    print(f"{'='*70}")
    
    # 1. Cargar Dataset
    ds_config = {"type": dataset_name, "test_size": 0.2, "seed": SEED}
    try:
        ds = DatasetFactory.load_from_config(ds_config, project_root=ROOT)
    except Exception as e:
        print(f"❌ Error cargando dataset {dataset_name}: {e}")
        return None
    
    # 2. Configuración Base
    base_config = {
        'federation': {'n_clients': 3, 'distribution': 'iid', 'seed': SEED},
        'model': {'n_estimators': 100, 'split_criterion': 'entropy'},
        'aggregation': {'strategy': STRATEGY},
        'prediction': {},
        'seed': SEED
    }
    
    # 3. Ejecutar Optimizador
    optimizer = HyperparamOptimizer(
        dataset_split=ds,
        strategy=STRATEGY,
        base_config=base_config,
        search_space=SEARCH_SPACE,
        verbose=False
    )
    
    study = optimizer.optimize(n_trials=N_TRIALS, metric=METRIC, seed=SEED)
    
    # 4. Guardar Resultados
    res = {
        'dataset': dataset_name,
        'best_value': study.best_value,
        'best_params': study.best_params,
        'n_trials': N_TRIALS
    }
    
    output_path = RESULTS_DIR / f"{STRATEGY.lower()}_{dataset_name.lower()}_results.json"
    with open(output_path, 'w') as f:
        json.dump(res, f, indent=2)
    
    print(f"✅ Resultados guardados en: {output_path.name}")
    return res

## 🧪 Ejecución por Dataset

Ejecuta las celdas de los datasets que desees optimizar.

In [ ]:
run_optimization('Iris')

In [ ]:
run_optimization('Car')

In [ ]:
run_optimization('Optdigits')

In [ ]:
run_optimization('Spambase')

In [ ]:
run_optimization('Nursery')

In [ ]:
run_optimization('Letter')

## 📊 Resumen de Resultados

In [ ]:
all_res = []
for f in RESULTS_DIR.glob(f"{STRATEGY.lower()}_*_results.json"):
    with open(f) as file:
        data = json.load(file)
        row = {'Dataset': data['dataset'], 'Best Macro-F1': data['best_value']}
        row.update(data['best_params'])
        all_res.append(row)

if all_res:
    df = pd.DataFrame(all_res)
    display(df.sort_values('Best Macro-F1', ascending=False))
else:
    print("⚠️ No hay resultados guardados aún.")